# Optimization by a Genetic Algorithm — evolving a sentence

*Utrecht University Molecular Modelling courses from the [Bonvin lab](https://bonvinlab.org).*

*Adapted from the GeeksforGeeks genetic-algorithm example (geeksforgeeks.org/genetic-algorithms).*

A minimal, self-contained **genetic algorithm (GA)**: a population of random strings is *evolved*
toward a target sentence by imitating natural selection. It is a third flavour of optimization to
sit alongside the **energy-minimization** notebooks (follow the gradient downhill) and the
**Monte Carlo** notebooks (sample by the Metropolis rule) — here the search is driven by
**evolution** of a whole population, and it needs no gradient at all.

## Theory in brief

A genetic algorithm keeps a **population** of candidate solutions and improves it generation by
generation with four ingredients borrowed from biology:

* **Chromosome** — one candidate solution. Here it is a list of characters (a string); the search
  space is every string of the target's length over the allowed alphabet `GENES`.
* **Fitness** — how good a candidate is. Here we use the number of characters that **differ** from
  the target, so **lower is better** and fitness $0$ is the perfect match (the global optimum).
* **Selection** — fitter individuals get to reproduce. We sort by fitness and draw parents from the
  fittest `PARENT_POOL` individuals.
* **Elitism, crossover, mutation** — the next generation is built by (i) copying the fittest
  `ELITE_FRAC` fraction unchanged (**elitism**, so the best is never lost), then (ii) **mating**
  pairs of parents: each child gene is copied from parent 1 or parent 2 (**crossover**) or, with
  probability `MUTATION_RATE`, replaced by a random gene (**mutation**, which maintains diversity
  and lets the search escape dead ends).

Repeating this loop, the population drifts toward ever-fitter strings until one matches the target.

### The role of the mutation rate
Mutation is a double-edged sword. **Too little** and the population loses diversity and stalls;
**too much** and it never settles — near the optimum, a high per-gene mutation rate keeps breaking
the almost-correct string faster than selection can fix it, producing a long convergence *tail*.
The original example hard-coded `MUTATION_RATE = 0.10`, which converges only after **~1000–3000**
generations; the default here, `0.05`, reaches the target in a **few hundred**. It is the single
most instructive knob in this notebook — try changing it.

### Swap mutation (optional)
A second mutation operator, **swap mutation**, exchanges two genes' *positions* in the chromosome
(`...ba...` → `...ab...`) rather than drawing a new random gene; it is controlled by `SWAP_RATE`
(default `0`). It has a crucial limitation: swapping only **re-orders the genes already present** —
it can never *introduce* a character that is missing. For this positional string-matching problem
that makes it almost useless on its own: with point mutation switched off (`MUTATION_RATE = 0`,
`SWAP_RATE > 0`) the search **never converges**, because there is no way to create the right letters.
So swap mutation only makes sense **in combination with point mutation**, where it adds a little
extra reshuffling. It genuinely pays off on **permutation / ordering problems** (e.g. the
travelling-salesman route), where a solution is an arrangement of a *fixed* set of items and point
mutation would create invalid duplicates.

**How much to swap?** Here we exchange a **single** pair of positions — the smallest, least
destructive swap. The swapped length is effectively a *step-size* knob: swapping longer contiguous
*blocks* only makes things worse for a positional target, because it displaces more genes that were
already correct. Empirically, convergence is unchanged up to a block length of ~2 and then degrades
steadily (a length-5 block roughly doubles the number of generations needed). Big, block-length swaps
are useful mainly on permutation problems (as *inversion* / 2-opt moves), not here.

## 1. Imports

The GA itself uses only the Python **standard library** (`random`). **matplotlib** is the one
third-party dependency, for the convergence plot and the animation (embedded inline via `jshtml`).
The cell installs matplotlib if it is missing (handy on Google Colab), then imports everything.

In [ ]:
# --- Install required packages if missing (e.g. on Google Colab) ---
import importlib.util, subprocess, sys

for pkg in ["matplotlib"]:
    if importlib.util.find_spec(pkg) is None:
        print(f"Installing {pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
    else:
        print(f"{pkg} already available")

import random

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib import rc

# Show animations inline
rc('animation', html='jshtml')
%matplotlib inline

## 2. The `Individual` class

One `Individual` is one candidate solution. It stores its `chromosome` (a list of characters) and
its `fitness`, and knows how to:

* `create_gnome` — build a fresh random chromosome (used to seed the initial population);
* `mate` — combine two parents into a child by crossover + mutation (see the theory above);
* `cal_fitness` — count how many characters differ from the target.

In [ ]:
class Individual:
    """One candidate solution in the population."""

    def __init__(self, chromosome):
        self.chromosome = chromosome
        self.fitness = self.cal_fitness()

    @classmethod
    def mutated_genes(cls):
        """A single random gene, used for mutation."""
        return random.choice(GENES)

    @classmethod
    def create_gnome(cls):
        """A fresh random chromosome the length of the target."""
        return [cls.mutated_genes() for _ in range(len(TARGET))]

    def mate(self, par2):
        """Produce one offspring by crossover + mutation."""
        # mutation happens with probability MUTATION_RATE; the parents split the rest equally
        parent_cut = (1.0 - MUTATION_RATE) / 2.0
        child_chromosome = []
        for gp1, gp2 in zip(self.chromosome, par2.chromosome):
            prob = random.random()
            if prob < parent_cut:
                child_chromosome.append(gp1)                    # gene from parent 1
            elif prob < 2 * parent_cut:
                child_chromosome.append(gp2)                    # gene from parent 2
            else:
                child_chromosome.append(self.mutated_genes())   # mutate (diversity)

        # optional swap mutation: exchange two gene positions. This only re-orders genes
        # that are already present (it cannot introduce a missing character), so it never
        # converges alone -- it only helps together with point mutation.
        if SWAP_RATE > 0 and random.random() < SWAP_RATE:
            i = random.randrange(len(child_chromosome))
            j = random.randrange(len(child_chromosome))
            child_chromosome[i], child_chromosome[j] = child_chromosome[j], child_chromosome[i]

        return Individual(child_chromosome)

    def cal_fitness(self):
        """Fitness = number of characters that differ from the target (0 is perfect)."""
        return sum(1 for gs, gt in zip(self.chromosome, TARGET) if gs != gt)

## 3. Parameters

Change any of these and re-run **this cell together with the Run cell just below**. `MUTATION_RATE`
is the one to experiment with first (see the theory); `TARGET` can be any sentence built from the
characters in `GENES`.

| Parameter | Meaning | Typical value |
|---|---|---|
| `TARGET` | the sentence to evolve towards | any string over `GENES` |
| `POPULATION_SIZE` | individuals per generation | 200 |
| `GENES` | the allowed alphabet | letters, digits, punctuation |
| `MUTATION_RATE` | per-gene mutation probability | 0.05 (original used 0.10) |
| `SWAP_RATE` | probability of a position-swap mutation per child (0 = off) | 0.0 |
| `ELITE_FRAC` | fraction copied unchanged each generation | 0.10 |
| `PARENT_POOL` | mating draws parents from the fittest this many | 50 |
| `SEED` | random-number seed (reproducibility) | 100 |
| `MAX_GENERATIONS` | safety cap on the number of generations | 2000 |

In [ ]:
TARGET = "Structural bioinformatics is the best master course!"

POPULATION_SIZE = 200

# allowed alphabet (a single string -- must NOT contain a line break)
GENES = (
    "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOP"
    'QRSTUVWXYZ 1234567890, .-;:_!"#%&/()=?@${[]}'
)

MUTATION_RATE   = 0.05   # per-gene mutation probability (original GeeksforGeeks value: 0.10 -> very slow)
SWAP_RATE       = 0.0    # optional swap mutation (exchange two positions); 0 = off. Only useful WITH point mutation
ELITE_FRAC      = 0.10   # fraction of fittest individuals copied unchanged each generation
PARENT_POOL     = 50     # parents are drawn from the fittest this many individuals
SEED            = 100    # random-number seed
MAX_GENERATIONS = 2000   # safety cap

## 4. Run the genetic algorithm

This loop replaces the script's `main()`. It seeds a random population, then each generation sorts
by fitness, keeps the elite, and breeds the rest, recording the **best** and **mean** fitness and
the best string so we can plot and animate the evolution. It stops when the best fitness reaches
$0$ (an exact match) or after `MAX_GENERATIONS`.

In [ ]:
def run_ga():
    random.seed(SEED)
    population = [Individual(Individual.create_gnome()) for _ in range(POPULATION_SIZE)]

    best_fit = []; mean_fit = []; best_str = []

    for generation in range(1, MAX_GENERATIONS + 1):
        population.sort(key=lambda ind: ind.fitness)         # fittest (lowest) first
        best = population[0]
        best_fit.append(best.fitness)
        mean_fit.append(sum(ind.fitness for ind in population) / len(population))
        best_str.append("".join(best.chromosome))

        if best.fitness <= 0:                                # exact match found
            break

        n_elite = int(ELITE_FRAC * POPULATION_SIZE)
        new_generation = population[:n_elite]                # elitism
        while len(new_generation) < POPULATION_SIZE:         # breed the rest
            p1 = random.choice(population[:PARENT_POOL])
            p2 = random.choice(population[:PARENT_POOL])
            new_generation.append(p1.mate(p2))
        population = new_generation

        if generation % 20 == 0:
            print("generation %4d   fitness %2d   %s" % (generation, best.fitness, best_str[-1]))

    print("\nConverged at generation %d (fitness %d):" % (len(best_fit), best_fit[-1]))
    print("  ", best_str[-1])
    return best_fit, mean_fit, best_str


best_fit, mean_fit, best_str = run_ga()

## 5. Convergence

The **best** fitness in the population (blue) falls to $0$ when the target is found; the **mean**
fitness (orange) settles at a higher value because ongoing mutation keeps injecting errors across
the population. Notice the characteristic shape: a fast initial drop as the easy characters lock in,
then a slower approach as the last few are pinned down (the *tail* set by `MUTATION_RATE`).

In [ ]:
gens = range(1, len(best_fit) + 1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(gens, best_fit, lw=1.8, color="tab:blue",   label="best fitness")
ax.plot(gens, mean_fit, lw=1.2, color="tab:orange", label="mean fitness")
ax.axhline(0, ls="--", color="black", lw=1)
ax.set_xlabel("generation")
ax.set_ylabel("fitness  (characters wrong; lower is better)")
ax.set_title(f"GA convergence  —  solved in {len(best_fit)} generations (mutation {MUTATION_RATE})")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 6. Watch the sentence evolve

The best string in each generation, character by character: **green** where it already matches the
target, **red** where it does not. Red letters turn green as the population evolves.
(Only ~120 frames are shown — adjust `stride`.)

In [ ]:
L = len(TARGET)
stride = max(1, len(best_str) // 120)
frames = list(range(0, len(best_str), stride))
if frames[-1] != len(best_str) - 1:
    frames.append(len(best_str) - 1)   # always show the final (solved) frame

fig, ax = plt.subplots(figsize=(13, 1.8))
ax.set_xlim(-0.5, L - 0.5)
ax.set_ylim(-1, 1)
ax.axis('off')

char_texts = []
for i in range(L):
    t = ax.text(i, 0, best_str[0][i], ha='center', va='center',
                family='monospace', fontsize=13, fontweight='bold')
    char_texts.append(t)
gen_title = ax.set_title("")

def update(frame_idx):
    f = frames[frame_idx]
    s = best_str[f]
    for i, t in enumerate(char_texts):
        t.set_text(s[i])
        t.set_color("#1a9850" if s[i] == TARGET[i] else "#d73027")
    gen_title.set_text(f"generation {f+1}   fitness {best_fit[f]}   ({L - best_fit[f]}/{L} correct)")
    return char_texts + [gen_title]

anim = FuncAnimation(fig, update, frames=len(frames), interval=100, blit=False)
plt.close(fig)   # avoid a duplicate static figure
anim

## 7. Genetic algorithms as optimization

This toy problem has an obvious answer (we *know* the target), which makes it perfect for *seeing*
how a genetic algorithm works — but the same machinery optimizes problems where the answer is
**not** known in advance, wherever candidate solutions can be encoded as "chromosomes" and scored by
a fitness function. In structural biology that includes protein and ligand design, and
**docking** (evolving the position/orientation/conformation of a molecule to optimise a binding
score).

Put alongside the other optimizers in this course:

* **Energy minimization** follows the local **gradient** downhill — fast, but it only finds the
  nearest minimum.
* **Monte Carlo / Metropolis** **samples** configurations at a temperature, and can cross barriers.
* **A genetic algorithm** evolves a **whole population** by selection + crossover + mutation. It
  uses **no gradient**, explores many regions at once, and — thanks to mutation and recombination —
  can escape local optima, at the cost of many fitness evaluations.

Things to try: raise `MUTATION_RATE` toward `0.10` and watch the convergence tail grow (or drop it
to `0.02`); turn on `SWAP_RATE` (say `0.3`) — it barely helps here, and setting `MUTATION_RATE = 0`
with `SWAP_RATE > 0` shows that swap-*only* never converges; shrink `PARENT_POOL` or `ELITE_FRAC` to
change the selection pressure; or change `TARGET` to your own sentence.